

### 2.1 理论计算题：一阶马尔可夫模型 + 拉普拉斯平滑

**给定序列：** `"ababc"`，词汇表 `V = {'a', 'b', 'c'}`

**拉普拉斯平滑（加1平滑）公式：**
$$p(x_t | x_{t-1}) = \frac{\text{count}(x_{t-1}, x_t) + 1}{\text{count}(x_{t-1}) + |V|}$$

**统计转移次数：**

| 转移 | 次数 |
|:---:|:---:|
| a→b | 1 |
| b→a | 1 |
| b→b | 0 |
| b→c | 1 |
| c→(无) | 0 |

**各状态出现次数：** count(a)=2, count(b)=2, count(c)=1

---

#### 1. $p('a' | 'b')$

$$\text{count}(b \to a) = 1, \quad \text{count}(b) = 2, \quad |V| = 3$$

$$p('a'|'b') = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

---

#### 2. $p('c' | 'b')$

$$\text{count}(b \to c) = 1, \quad \text{count}(b) = 2, \quad |V| = 3$$

$$p('c'|'b') = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

---

### 3.1 理论计算题：线性RNN的BPTT梯度推导

**模型定义：**
$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$
$$o_t = W_{oh} h_t$$
$$L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2$$

**推导 $\frac{\partial L}{\partial W_{hh}}$：**

定义 $\delta_t = \frac{\partial L}{\partial h_t}$，由链式法则：

$$\delta_t = \frac{\partial L}{\partial o_t} \frac{\partial o_t}{\partial h_t} + \frac{\partial L}{\partial h_{t+1}} \frac{\partial h_{t+1}}{\partial h_t} = W_{oh}^T (o_t - y_t) + W_{hh}^T \delta_{t+1}$$

（边界条件：$\delta_{T+1} = 0$）

对 $W_{hh}$ 的梯度：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t h_{t-1}^T$$

展开后，$\delta_t$ 包含从 $t$ 到 $T$ 的反向传播：

$$\delta_t = \sum_{s=t}^{T} (W_{hh}^T)^{s-t} W_{oh}^T (o_s - y_s)$$

因此：

$$\boxed{\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \left( \sum_{s=t}^{T} (W_{hh}^T)^{s-t} W_{oh}^T (o_s - y_s) \right) h_{t-1}^T}$$

**梯度消失/爆炸条件：**

对 $W_{hh}$ 做特征值分解，设其特征值为 $\lambda_i$。当反向传播经过 $k$ 个时间步时，梯度中包含 $W_{hh}^k$。

- **梯度爆炸：** 当 $|\lambda_{\max}| > 1$ 时，$(W_{hh})^k$ 随 $k$ 指数增长
- **梯度消失：** 当 $|\lambda_{\max}| < 1$ 时，$(W_{hh})^k$ 随 $k$ 指数衰减至0

---

### 4.1 理论计算题：深度双向RNN参数总数

**模型结构：** $L$ 层，每层 $H$ 个隐藏单元，输入维度 $D$，输出维度 $O$

**第1层（双向）：**
- 前向RNN：输入→隐藏 $W_{hx}^{(f)} \in \mathbb{R}^{H \times D}$，隐藏→隐藏 $W_{hh}^{(f)} \in \mathbb{R}^{H \times H}$，偏置 $b_h^{(f)} \in \mathbb{R}^{H}$
- 后向RNN：输入→隐藏 $W_{hx}^{(b)} \in \mathbb{R}^{H \times D}$，隐藏→隐藏 $W_{hh}^{(b)} \in \mathbb{R}^{H \times H}$，偏置 $b_h^{(b)} \in \mathbb{R}^{H}$

第1层参数：$2 \times (HD + H^2 + H) = 2HD + 2H^2 + 2H$

**第 $l$ 层（$l \geq 2$，双向，输入为前一层拼接后的 $2H$ 维）：**
- 每层参数：$2 \times (H \cdot 2H + H^2 + H) = 4H^2 + 2H^2 + 2H = 6H^2 + 2H$

第2~L层总参数：$(L-1)(6H^2 + 2H)$

**输出层（仅最后输出层，从拼接隐藏状态 $2H$ 维映射到 $O$ 维）：**
- 权重 $W_{out} \in \mathbb{R}^{O \times 2H}$，偏置 $b_{out} \in \mathbb{R}^{O}$
- 参数：$2HO + O$

**总参数数：**

$$\boxed{N_{total} = 2HD + 2H^2 + 2H + (L-1)(6H^2 + 2H) + 2HO + O}$$

简化：
$$N_{total} = 2HD + (6L - 4)H^2 + 2LH + 2HO + O$$

---

### 5.1 理论计算题：Skip-gram + 负采样

**模型设定：**
- 中心词 $w_c$，词向量 $v_c$
- 上下文词 $w_o$，词向量 $u_o$
- 负样本词 $w_{n_k}$（$k=1,\dots,K$），词向量 $u_{n_k}$
- 噪声分布 $P_n(w)$（通常取unigram分布的3/4次幂）

**目标函数（对数似然）：**

对于正样本 $(w_c, w_o)$ 和 $K$ 个负样本，使用sigmoid函数 $\sigma(x) = \frac{1}{1+e^{-x}}$：

$$\boxed{\mathcal{L} = \log \sigma(u_o^T v_c) + \sum_{k=1}^{K} \mathbb{E}_{w_{n_k} \sim P_n(w)} \left[ \log \sigma(-u_{n_k}^T v_c) \right]}$$

或写成完整形式：

$$\mathcal{L} = \log \frac{1}{1 + \exp(-u_o^T v_c)} + \sum_{k=1}^{K} \log \frac{1}{1 + \exp(u_{n_k}^T v_c)}$$

**负样本采样方法：**

1. 计算每个词 $w$ 的噪声分布概率：$P_n(w) = \frac{f(w)^{3/4}}{\sum_{w'} f(w')^{3/4}}$，其中 $f(w)$ 是词 $w$ 在语料中的频率
2. 将词汇表按 $P_n(w)$ 累积概率划分区间
3. 从均匀分布中采样，根据所在区间选择对应的负样本词

---

### 6.1 理论计算题：缩放点积注意力

**给定：**
- $Q \in \mathbb{R}^{2 \times 4}$，$K \in \mathbb{R}^{3 \times 4}$，$V \in \mathbb{R}^{3 \times 5}$
- $d_k = 4$，$\sqrt{d_k} = 2$

**计算步骤：**

**Step 1: 计算得分矩阵 $S = QK^T / \sqrt{d_k}$**

$$S \in \mathbb{R}^{2 \times 3}$$

设 $Q = \begin{bmatrix} q_1 \\ q_2 \end{bmatrix}$，$K = \begin{bmatrix} k_1 \\ k_2 \\ k_3 \end{bmatrix}$，则：

$$S_{ij} = \frac{q_i \cdot k_j}{2}$$

**Step 2: Softmax（按行）**

$$A_{ij} = \frac{\exp(S_{ij})}{\sum_{j'=1}^{3} \exp(S_{ij'})}$$

注意力权重矩阵 $A \in \mathbb{R}^{2 \times 3}$，每行和为1。

**Step 3: 加权求和**

$$\text{Output} = AV \in \mathbb{R}^{2 \times 5}$$

$$\text{Output}_{i,:} = \sum_{j=1}^{3} A_{ij} \cdot v_j$$

其中 $v_j$ 是 $V$ 的第 $j$ 行。

**最终输出：** $\boxed{\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V \in \mathbb{R}^{2 \times 5}}$

---


### 2.2 编程题

In [7]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数
    
    参数:
        text: 输入文本字符串
        n: 滑动窗口大小
    
    返回:
        vocab: 词汇表字典 {word: id}
        (features, labels): 特征列表和标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    words = [w for w in words if w]
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.keys(), key=lambda w: (-word_counts[w], w))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    
    for i in range(len(words) - n):
        feature = words[i:i + n]
        label = words[i + n]
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)

### 3.2 编程题

In [8]:
import numpy as np

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单步前向传播（使用tanh激活）
    
    参数:
        x_t: 输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏权重，形状 (input_size, hidden_size)
        W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
        b_h: 隐藏层偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态
        cache: 缓存用于反向传播
    """
    linear = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    h_t = np.tanh(linear)
    cache = (x_t, h_prev, W_hx, W_hh, linear, h_t)
    return h_t, cache


def rnn_step_backward(dh_next, cache):
    """
    RNN单步反向传播
    
    参数:
        dh_next: 上游梯度，损失对h_t的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存
    
    返回:
        dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    x_t, h_prev, W_hx, W_hh, linear, h_t = cache
    batch_size = x_t.shape[0]
    
    # tanh导数: 1 - tanh^2(x)
    dtanh = dh_next * (1 - h_t ** 2)
    
    dx_t = np.dot(dtanh, W_hx.T)
    dh_prev = np.dot(dtanh, W_hh.T)
    dW_hx = np.dot(x_t.T, dtanh)
    dW_hh = np.dot(h_prev.T, dtanh)
    db_h = np.sum(dtanh, axis=0)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

### 4.2 编程题

In [9]:
import torch
import torch.nn as nn

def bidirectional_rnn_encoder(X, input_dim, hidden_dim, num_layers=1):
    """
    双向RNN编码器
    
    参数:
        X: 输入序列，形状 (seq_len, batch, input_dim)
        input_dim: 输入维度
        hidden_dim: 隐藏层维度
        num_layers: RNN层数
    
    返回:
        outputs: 每个时间步拼接后的隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
        final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
    """
    rnn = nn.RNN(
        input_size=input_dim,
        hidden_size=hidden_dim,
        num_layers=num_layers,
        bidirectional=True,
        batch_first=False
    )
    
    outputs, hidden = rnn(X)
    
    # 提取最后一层的前向和后向最终隐藏状态
    if num_layers == 1:
        final_forward = hidden[0]
        final_backward = hidden[1]
    else:
        final_forward = hidden[num_layers - 1]
        final_backward = hidden[2 * num_layers - 1]
    
    final_state = torch.cat([final_forward, final_backward], dim=-1)
    
    return outputs, final_state

### 5.2 编程题

In [10]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_index, W_in, W_out):
    """
    CBOW模型前向传播和损失计算（完整softmax，无负采样）
    
    参数:
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        target_index: 中心词目标索引，形状 (batch_size,)
        W_in: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 交叉熵损失值（标量）
    """
    # 获取嵌入向量: (batch_size, context_size, d)
    context_embeds = W_in[context_indices]
    
    # 平均上下文向量作为隐藏层: (batch_size, d)
    h = torch.mean(context_embeds, dim=1)
    
    # 计算输出分数: (batch_size, V)
    scores = torch.matmul(h, W_out)
    
    # 交叉熵损失
    loss = F.cross_entropy(scores, target_index)
    
    return loss

### 6.2 编程题

In [11]:
import torch
import torch.nn as nn
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """缩放点积注意力"""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attention_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    return output, attention_weights


def multi_head_attention(X, num_heads=2, d_model=4):
    """
    多头注意力前向传播
    
    参数:
        X: 输入，形状 (seq_len, batch, d_model)
        num_heads: 注意力头数
        d_model: 模型维度
    
    返回:
        output: 输出，形状 (seq_len, batch, d_model)
    """
    assert d_model % num_heads == 0
    seq_len, batch_size, _ = X.shape
    d_k = d_v = d_model // num_heads
    
    # 线性投影层
    W_q = nn.Linear(d_model, d_model)
    W_k = nn.Linear(d_model, d_model)
    W_v = nn.Linear(d_model, d_model)
    W_o = nn.Linear(d_model, d_model)
    
    # 线性投影
    Q = W_q(X).transpose(0, 1)  # (batch, seq_len, d_model)
    K = W_k(X).transpose(0, 1)
    V = W_v(X).transpose(0, 1)
    
    # 重塑为多头: (batch, num_heads, seq_len, d_k)
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, d_v).transpose(1, 2)
    
    # 合并多头计算
    Q_reshaped = Q.reshape(batch_size * num_heads, seq_len, d_k)
    K_reshaped = K.reshape(batch_size * num_heads, seq_len, d_k)
    V_reshaped = V.reshape(batch_size * num_heads, seq_len, d_v)
    
    attn_output, _ = scaled_dot_product_attention(Q_reshaped, K_reshaped, V_reshaped)
    
    # 恢复并拼接
    attn_output = attn_output.view(batch_size, num_heads, seq_len, d_v)
    attn_output = attn_output.transpose(1, 2).reshape(batch_size, seq_len, d_model)
    
    # 最终线性层
    output = W_o(attn_output.transpose(0, 1))
    
    return output